In [ ]:
ReportFolderName = 'GPT-Based_Base_Version'
model_alias = "Gemma3"
random_state = 44

In [ ]:
import shutil
import os

# Keywords for folders to delete
folders_to_delete = ["logs", ReportFolderName, "results", "sample_data"]

# Delete matching folders
for item in os.listdir("."):
    if os.path.isdir(item) and any(keyword in item for keyword in folders_to_delete):
        shutil.rmtree(item)
        print(f"✅ Deleted folder: {item}")

# Delete all files in the current directory
for item in os.listdir("."):
    if os.path.isfile(item):
        os.remove(item)
        print(f"🗑️ Deleted file: {item}")

print("\n🎯 Full cleanup completed. All matching folders and all files removed.")

✅ Deleted folder: sample_data

🎯 Full cleanup completed. All matching folders and all files removed.


In [ ]:
reports_dir = f"{ReportFolderName}"
os.makedirs(reports_dir, exist_ok=True)

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl
!pip install -q scikit-learn pandas numpy tqdm

import os
os.environ["WANDB_MODE"] = "disabled"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 53.6 MB/s eta 0:00:00


In [ ]:

# ============================================
# STEP 1: INSTALL DEPENDENCIES
# ============================================
# Uncomment these lines if running in a new Colab environment
# !pip install -q -U transformers datasets peft accelerate bitsandbytes trl
# !pip install -q scikit-learn pandas numpy tqdm

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict,Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm

# ============================================
# STEP 2: CONFIGURATION
# ============================================
MODEL_ID = "google/gemma-3-4b-it"
VALID_LABELS = ["smish", "promo", "normal"]

In [ ]:
# ============================================
# STEP 3: LOAD DATA & PREPARE SPLITS
# ============================================
print("\n📥 Loading SMS dataset...")
dataset = load_dataset("shariul-islam/bengali-sms-smishing-dataset")

# Only for two variant
variants = ["Bengali", "English"]
train_dataset_filtered = dataset["train"].filter(lambda example: example["source"] in variants)
dataset = DatasetDict({
    "train": train_dataset_filtered,
    "validation": dataset["validation"],
    "test": dataset["test"]
})

# Shuffle and select samples
all_data = dataset['train'].shuffle(seed=42)

train_samples = dataset['train']
test_samples = dataset['test']
validation_samples = dataset['validation']

# Select Train (100) and Test (20) samples
# train_samples = all_data.select(range(100))
# test_samples = all_data.select(range(100, 120))
# validation_samples = all_data.select(range(100, 110))


print(f"✅ Train samples: {len(train_samples)}")
print(f"✅ Test samples: {len(test_samples)}")
print(f"✅ validation samples: {len(validation_samples)}")



📥 Loading SMS dataset...


README.md:   0%|          | 0.00/554 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/447k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/59.2k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/112k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5604 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/700 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5604 [00:00<?, ? examples/s]

✅ Train samples: 2820
✅ Test samples: 1401
✅ validation samples: 700


In [ ]:

# ============================================
# STEP 4: PROMPT TEMPLATES & HELPERS
# ============================================
def get_zero_shot_prompt(sms: str) -> str:
    return f'''You are an expert in SMS content classification for fraud detection and marketing analysis.

SMS: "{sms}"

Task: Classify the above SMS message into one of three categories:
1. smish — Fraudulent or scam SMS that tries to trick users into revealing sensitive information, clicking malicious links, or calling scam numbers.
2. promo — Promotional or marketing SMS offering discounts, sales, cashback, or advertisements.
3. normal — Regular personal messages, greetings, casual conversations.

Instructions:
- Response will only be either smish, promo, or normal.
- A single-word response.

Response:'''

def get_training_prompt(sms: str, label: str) -> str:
    return f"{get_zero_shot_prompt(sms)} {label}"

def prepare_training_data(samples) -> Dataset:
    texts = []
    for sample in samples:
        prompt = get_training_prompt(sample['text'], sample['label'])
        texts.append({"text": prompt})
    return Dataset.from_list(texts)

def classify_sms(model, tokenizer, sms_text: str) -> str:
    """Classify a single SMS message"""
    prompt = get_zero_shot_prompt(sms_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    response = generated.strip().lower()

    # Simple mapping to handle extra punctuation
    if 'smish' in response: return 'smish'
    if 'promo' in response: return 'promo'
    if 'normal' in response: return 'normal'
    return "unknown"

def classify_sms_with_probs(model, tokenizer, sms_text: str) -> tuple:
    """Classify SMS with probability scores for each class"""
    prompt = get_zero_shot_prompt(sms_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Get the token IDs for our labels
    # We check the first token of each label
    smish_token = tokenizer.encode("smish", add_special_tokens=False)[0]
    promo_token = tokenizer.encode("promo", add_special_tokens=False)[0]
    normal_token = tokenizer.encode("normal", add_special_tokens=False)[0]

    with torch.no_grad():
        # Get logits instead of generating
        outputs = model(**inputs)

        # Get logits for the next token (last position)
        next_token_logits = outputs.logits[0, -1, :]

        # Apply softmax to get probabilities
        probs = torch.softmax(next_token_logits, dim=-1)

        # Extract probabilities for our target labels
        prob_smish = probs[smish_token].item()
        prob_promo = probs[promo_token].item()
        prob_normal = probs[normal_token].item()

        # Normalize to sum to 1 (since we only care about these 3)
        total = prob_smish + prob_promo + prob_normal
        prob_smish_norm = prob_smish / total
        prob_promo_norm = prob_promo / total
        prob_normal_norm = prob_normal / total

        # Get predicted class
        probs_dict = {
            'smish': prob_smish_norm,
            'promo': prob_promo_norm,
            'normal': prob_normal_norm
        }
        predicted = max(probs_dict, key=probs_dict.get)

    return predicted, probs_dict



In [ ]:
# ============================================
# STEP 5: LOAD BASE MODEL (QUANTIZED)
# ============================================
print("\n⚙️ Loading Base Model (4-bit)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16, # Explicitly set this
    trust_remote_code=True,
)


⚙️ Loading Base Model (4-bit)...


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
# ============================================
# STEP 6: EVALUATE ZERO-SHOT (BASELINE)
# ============================================
print("\n" + "="*50)
print("🔍 EVALUATION 1: ZERO-SHOT BASELINE")
print("="*50)
print("Running inference on test_samples BEFORE training...")

y_true = [sample['label'] for sample in test_samples]
y_pred_zero = []
analysis_results = []

for sample in tqdm(test_samples, desc="Zero-Shot Inference"):
    pred, probs = classify_sms_with_probs(base_model, tokenizer, sample['text'])
    y_pred_zero.append(pred)

    # Store all relevant info in a dictionary
    analysis_results.append({
        "SMS_Text": sample['text'],
        "True_Label": sample['label'],
        "Predicted_Label": pred,
        "Prob_Smish": probs.get('smish', 0),
        "Prob_Promo": probs.get('promo', 0),
        "Prob_Normal": probs.get('normal', 0),
        "Source": sample['source'],
        "Is_Correct": 1 if pred == sample['label'] else 0
    })
    print({
            "SMS_Text": sample['text'],
            "True_Label": sample['label'],
            "Predicted_Label": pred,
            "Prob_Smish": probs.get('smish', 0),
            "Prob_Promo": probs.get('promo', 0),
            "Prob_Normal": probs.get('normal', 0),
            "Source": sample['source'],
            "Is_Correct": 1 if pred == sample['label'] else 0
        })

# Create DataFrame
df_analysis = pd.DataFrame(analysis_results)

# Save to CSV
csv_filename = "smish_detection_base_line_results.csv"
df_analysis.to_csv(f"{reports_dir}/{csv_filename}", index=False, encoding='utf-8-sig')

# Store baseline metrics
acc_zero = accuracy_score(y_true, y_pred_zero)
prec_zero, rec_zero, f1_zero, _ = precision_recall_fscore_support(y_true, y_pred_zero, average='weighted', zero_division=0)

print(f"\nBaseline Accuracy: {acc_zero:.4f}")
print("Baseline Classification Report:")
print(classification_report(y_true, y_pred_zero, zero_division=0))


🔍 EVALUATION 1: ZERO-SHOT BASELINE
Running inference on test_samples BEFORE training...


Zero-Shot Inference:   0%|          | 2/1401 [00:01<10:29,  2.22it/s]

{'SMS_Text': 'E-comerch Platform ( No Fee )\r\nআমাদের কোম্পানিতে কাজ করে প্রতিদিন $10/$15 USDT আয় করতে চাইলে Inbox', 'True_Label': 'smish', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9999918905227055, 'Prob_Promo': 4.801446336037328e-07, 'Prob_Normal': 7.629332660848278e-06, 'Source': 'Bengali', 'Is_Correct': 1}
{'SMS_Text': 'ব্যালেন্স শেষ? ২০০টাকা পর্যন্ত জিপি ইমার্জেন্সি ব্যালেন্স নিতে *৯#', 'True_Label': 'promo', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9146957549879725, 'Prob_Promo': 0.08492247932720284, 'Prob_Normal': 0.0003817656848246287, 'Source': 'Bengali', 'Is_Correct': 0}


Zero-Shot Inference:   0%|          | 4/1401 [00:01<05:32,  4.20it/s]

{'SMS_Text': 'Shwapno grocery mega offer: Weekly shopping e buy 1 get 1 free on fruits and vegetables. Offer across Dhaka outlets. Limited stock, hurry!', 'True_Label': 'promo', 'Predicted_Label': 'promo', 'Prob_Smish': 0.0011730527782105446, 'Prob_Promo': 0.9985025248128157, 'Prob_Normal': 0.00032442240897385374, 'Source': 'Banglish', 'Is_Correct': 1}
{'SMS_Text': 'GP Binge pack: Unlimited streaming internet 60GB only 499TK. Validity 15 days. Activate with *121*499# now.', 'True_Label': 'promo', 'Predicted_Label': 'promo', 'Prob_Smish': 0.002472419412238476, 'Prob_Promo': 0.9973590065369271, 'Prob_Normal': 0.00016857405083444156, 'Source': 'Banglish', 'Is_Correct': 1}


...

{'SMS_Text': 'কেমন চলছে তোমার দিন?', 'True_Label': 'normal', 'Predicted_Label': 'normal', 'Prob_Smish': 0.0004562283227134025, 'Prob_Promo': 5.664171884462664e-07, 'Prob_Normal': 0.9995432052600981, 'Source': 'Bengali', 'Is_Correct': 1}
{'SMS_Text': "There's a problem with Janata Bank account. Click here for details: https://t.me/JanataErrorBot", 'True_Label': 'smish', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9999980138755294, 'Prob_Promo': 4.920838806876096e-08, 'Prob_Normal': 1.9369160824811826e-06, 'Source': 'English', 'Is_Correct': 1}


Zero-Shot Inference: 100%|█████████▉| 1400/1401 [03:09<00:00,  7.47it/s]

{'SMS_Text': '8,000 taka biniyoge 24,000 taka pan! Call korun: +8801919900112', 'True_Label': 'smish', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9999991585745548, 'Prob_Promo': 1.487939515394441e-07, 'Prob_Normal': 6.926314937157749e-07, 'Source': 'Banglish', 'Is_Correct': 1}
{'SMS_Text': '40Tk chil cashback e 35GB+500min.@459Tk 30din, nao: cutt.ly/0wO6Jggn', 'True_Label': 'promo', 'Predicted_Label': 'promo', 'Prob_Smish': 0.2820946233194685, 'Prob_Promo': 0.7174258321458088, 'Prob_Normal': 0.00047954453472276546, 'Source': 'Banglish', 'Is_Correct': 1}


Zero-Shot Inference: 100%|██████████| 1401/1401 [03:09<00:00,  7.37it/s]

{'SMS_Text': 'Critical security breach at your BASIC Bank account. Act: basicbank-security.net', 'True_Label': 'smish', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9999983790356864, 'Prob_Promo': 4.144378739137952e-08, 'Prob_Normal': 1.5795205261972958e-06, 'Source': 'English', 'Is_Correct': 1}

Baseline Accuracy: 0.7580
Baseline Classification Report:
              precision    recall  f1-score   support

      normal       0.96      0.56      0.71       498
       promo       0.95      0.66      0.78       342
       smish       0.64      0.99      0.77       561

    accuracy                           0.76      1401
   macro avg       0.85      0.74      0.76      1401
weighted avg       0.83      0.76      0.75      1401



In [ ]:
# ============================================
# STEP 7: PREPARE FOR FINE-TUNING
# ============================================
print("\n🔧 Preparing model for LoRA training...")
base_model = prepare_model_for_kbit_training(base_model)



def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Get the highest probability token for every position
    predictions = np.argmax(logits, axis=-1)

    last_token_preds = []
    last_token_labels = []

    # Loop through the batch to find the specific classification token
    for i in range(labels.shape[0]):
        # Identify non-padding indices
        valid_indices = np.where(labels[i] != -100)[0]
        if len(valid_indices) > 0:
            last_idx = valid_indices[-1]
            # Logits at [last_idx - 1] predict the label at [last_idx]
            last_token_preds.append(predictions[i, last_idx - 1])
            last_token_labels.append(labels[i, last_idx])

    # Calculate classification-only metrics
    acc = accuracy_score(last_token_labels, last_token_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        last_token_labels, last_token_preds, average='weighted', zero_division=0
    )

    return {
        'eval_accuracy': acc,
        'eval_f1': f1,
        'eval_precision': precision,
        'eval_recall': recall
    }


# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# base_model.gradient_checkpointing_enable()
lora_config = LoraConfig(
    r=8,                          # Increased from 16 for higher linguistic capacity
    lora_alpha=32,                 # 2x Rank is the best practice
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,              # Added dropout to prevent memorizing specific phone numbers
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
train_dataset = prepare_training_data(train_samples)
val_dataset = prepare_training_data(validation_samples) # Process validation too


🔧 Preparing model for LoRA training...


In [ ]:
# ============================================
# STEP 8: TRAIN (FINE-TUNE)
# ============================================
print("\n🚀 Starting LoRA Fine-Tuning...")
training_args = SFTConfig(
    output_dir="./smishdetect-lora",
    num_train_epochs=10,               # 2 epochs is enough for 5.6k classification samples
    # per_device_train_batch_size=8,    # Increased for Colab Pro+ GPU (A100/L4)
    per_device_train_batch_size=32,    # Increased for Colab Pro+ GPU (A100/L4)
    # gradient_accumulation_steps=2,    # Effective Batch Size = 16
    # learning_rate=1e-4,               # Stable learning rate for Gemma
    learning_rate=3e-5,               # Stable learning rate for Gemma
    # max_length=128,               # SMS are short; 256 is plenty and saves VRAM
    weight_decay=0.01,                # Regularization to keep accuracy high on test data
    bf16=True,                        # Use bfloat16 for speed on modern GPUs

    # # Validation & Logging
    # eval_strategy="steps",            # Evaluate every X steps
    # eval_steps=100,                   # Check validation performance often
    # save_strategy="steps",
    # save_steps=100,
    # logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    metric_for_best_model="eval_f1",
    greater_is_better=True,

    # Optimization
    # optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",       # Smoothly lowers learning rate for better accuracy
    # warmup_ratio=0.1,                 # 10% of training spent "warming up"

    load_best_model_at_end=True,      # Automatically keep the most accurate version
    report_to="none",
)
training_args = SFTConfig(
    output_dir="./smishdetect-lora",
    num_train_epochs=3,
    # Maximize your L4/A100 GPU throughput
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    # max_length=256,              # Safe for prompt + SMS
    # packing=True,                    # Fills empty space for faster training
    bf16=True,                       # Fast 16-bit brain float

    # Strategy
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="eval_f1",  # Tracks the REAL metric
    load_best_model_at_end=True,

    # Faster Optimizer
    optim="adamw_torch_fused",
    report_to="none",
    seed = random_state
)
training_args = SFTConfig(
    output_dir="./smishdetect-lora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    seed=random_state,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
    # compute_metrics=compute_metrics
)

trainer.train()
print("\n💾 Saving adapter...")
# trainer.model.save_pretrained("./smishdetect-lora-adapter")


🚀 Starting LoRA Fine-Tuning...


Adding EOS to train dataset:   0%|          | 0/2820 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2820 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2820 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Step,Training Loss
10,1.441500
20,0.555400
30,0.504000
40,0.445400
50,0.463800
60,0.478500
70,0.484300
80,0.485500
90,0.434500
100,0.456100



💾 Saving adapter...


In [ ]:
trainer.model.save_pretrained("./gemma3-2v-fine-tune-adapter")
trainer.model.push_to_hub(
    repo_id="shariul-islam/gemma3-2v-fine-tune-adapter",
    private=False  # set True if you want private
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|1         |  638kB / 32.9MB            

CommitInfo(commit_url='https://huggingface.co/shariul-islam/gemma3-2v-fine-tune-adapter/commit/117be099df962b6203d37b0de3f6bc79b42fdf10', commit_message='Upload model', commit_description='', oid='117be099df962b6203d37b0de3f6bc79b42fdf10', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shariul-islam/gemma3-2v-fine-tune-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='shariul-islam/gemma3-2v-fine-tune-adapter'), pr_revision=None, pr_num=None)

In [ ]:
# Assuming you have already loaded your tokenizer
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")

# Encode the prompt to token IDs
prompt = get_training_prompt('How are you kjn klkl kll lml,m lml lmlmm', 'normal')
print(prompt)
tokens = tokenizer.encode(prompt, add_special_tokens=True)

# Count the tokens
token_count = len(tokens)

print(f"The prompt contains {token_count} tokens.")

You are an expert in SMS content classification for fraud detection and marketing analysis.

SMS: "How are you kjn klkl kll lml,m lml lmlmm"

Task: Classify the above SMS message into one of three categories:
1. smish — Fraudulent or scam SMS that tries to trick users into revealing sensitive information, clicking malicious links, or calling scam numbers.
2. promo — Promotional or marketing SMS offering discounts, sales, cashback, or advertisements.
3. normal — Regular personal messages, greetings, casual conversations.

Instructions:
- Response will only be either smish, promo, or normal.
- A single-word response.

Response: normal
The prompt contains 147 tokens.


In [ ]:
# ============================================
# STEP 9: EVALUATE FINE-TUNED MODEL
# ============================================
print("\n" + "="*50)
print("🔍 EVALUATION 2: FINE-TUNED MODEL")
print("="*50)
print("Running inference on test_samples AFTER training...")

# Note: 'model' is now the Fine-Tuned model (Base + LoRA)
y_pred = []
analysis_results = []
for sample in tqdm(test_samples, desc="Fine-Tuned Inference"):
    pred, probs = classify_sms_with_probs(model, tokenizer, sample['text'])
    y_pred.append(pred)

    # Store all relevant info in a dictionary
    analysis_results.append({
        "SMS_Text": sample['text'],
        "True_Label": sample['label'],
        "Predicted_Label": pred,
        "Prob_Smish": probs.get('smish', 0),
        "Prob_Promo": probs.get('promo', 0),
        "Prob_Normal": probs.get('normal', 0),
        "Source": sample['source'],
        "Is_Correct": 1 if pred == sample['label'] else 0
    })


# Create DataFrame
df_analysis = pd.DataFrame(analysis_results)

# Save to CSV
csv_filename = "smish_detection_fine_tuned_results.csv"
df_analysis.to_csv(f"{reports_dir}/{csv_filename}", index=False, encoding='utf-8-sig')

# Store fine-tuned metrics
acc_ft = accuracy_score(y_true, y_pred)
prec_ft, rec_ft, f1_ft, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"\nFine-Tuned Accuracy: {acc_ft:.4f}")
print("Fine-Tuned Classification Report:")
print(classification_report(y_true, y_pred, zero_division=0))



🔍 EVALUATION 2: FINE-TUNED MODEL
Running inference on test_samples AFTER training...


Fine-Tuned Inference:   0%|          | 0/1401 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Fine-Tuned Inference:  29%|██▉       | 403/1401 [01:23<03:28,  4.78it/s]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve,
    auc, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
import os

# --- Preparation for Metrics ---
labels = ['normal', 'promo', 'smish']
n_classes = len(labels)

# 1. GENERATE CLASSIFICATION REPORTS
report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df =  pd.DataFrame(report_dict).transpose().round(4)


# Text file
report_text = classification_report(y_true, y_pred)
with open(f"{reports_dir}/classification_report.txt", "w") as f:
        f.write(report_text)

# Save as CSV and LaTeX
report_df.to_csv(f"{reports_dir}/classification_report.csv")
report_df.to_latex(f"{reports_dir}/classification_report.tex", float_format="%.4f")

# 2. CONFUSION MATRIX (Overall)
cm = confusion_matrix(y_true, y_pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'Confusion Matrix - {model_alias}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig(f"{reports_dir}/{model_alias}_confusion_matrix.png")
plt.close()

# 3. CONFUSION MATRIX (Per-Source)
all_source_reports = []  # store metrics for summary
for source in df_analysis['Source'].unique():
    source_df = df_analysis[df_analysis['Source'] == source]
    y_true_src = source_df['True_Label']
    y_pred_src = source_df['Predicted_Label']
    cm_s = confusion_matrix(y_true_src, y_pred_src, labels=labels)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_s, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f'Confusion Matrix - {model_alias} ({source})')
    plt.savefig(f"{reports_dir}/{model_alias}_confusion_matrix_{source}.png")
    plt.close()

    # --- Classification Report ---
    report_dict = classification_report(
        y_true_src, y_pred_src,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose().round(4)
    report_df.to_csv(f"{reports_dir}/{model_alias}_classification_report_{source}.csv", index=True)

    # Add macro averages for summary
    all_source_reports.append({
        "source": source,
        "precision": round(report_dict["macro avg"]["precision"], 4),
        "recall": round(report_dict["macro avg"]["recall"], 4),
        "f1_score": round(report_dict["macro avg"]["f1-score"], 4)
        })
# --- Summary Report Across Sources ---
summary_df = pd.DataFrame(all_source_reports)
summary_df.loc["Average"] = summary_df.mean(numeric_only=True)
summary_df.to_csv(f"{reports_dir}/{model_alias}_source_summary_report.csv", index=False)

print("\n✅ Per-source classification reports saved.")
print(f"✅ Summary report saved to: {reports_dir}/{model_alias}_source_summary_report.csv")

# 4. ROC CURVE (One-vs-Rest)
# Convert y_true and probabilities to binarized format for multi-class ROC
y_true_bin = label_binarize(y_true, classes=labels)
y_score = df_analysis[['Prob_Normal', 'Prob_Promo', 'Prob_Smish']].values

plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'ROC {labels[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curve (One-vs-Rest)')
plt.legend(loc="lower right")
plt.savefig(f"{reports_dir}/roc_curve_combined.png")
plt.close()

# 5. APPEND TO SUMMARY CSV
summary_data = {
    "Model_Name": model_alias,
    "Accuracy": round(acc_ft, 4),
    "Precision": round(prec_ft, 4),
    "Recall": round(rec_ft, 4),
    "F1": round(f1_ft, 4),
    "Timestamp": pd.Timestamp.now()
}

summary_df = pd.DataFrame([summary_data])
summary_df.to_csv(f"{reports_dir}/summary.csv", index=False)

print(f"✅ Evaluation reports generated in: {reports_dir}")

In [ ]:
import shutil
from google.colab import files
import os

# List of folders to zip and download
folders_to_download = [
    ReportFolderName,
    #"stacking_ensemble_reports"
]

for folder in folders_to_download:
    if os.path.exists(folder):
        zip_filename = f"{folder}.zip"
        # Create zip archive
        shutil.make_archive(folder, 'zip', folder)
        # Download zip
        files.download(zip_filename)
        print(f"✅ Download started for '{zip_filename}'")
    else:
        print(f"⚠️ Folder '{folder}' not found")



In [ ]:
# ============================================
# STEP 10: COMPARISON RESULTS
# ============================================
print("\n" + "="*50)
print("📊 FINAL COMPARISON: ZERO-SHOT vs FINE-TUNED")
print("="*50)

results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (W)', 'Recall (W)', 'F1-Score (W)'],
    'Zero-Shot (Base)': [acc_zero, prec_zero, rec_zero, f1_zero],
    'Fine-Tuned (LoRA)': [acc_ft, prec_ft, rec_ft, f1_ft]
})

# Add Improvement Column
results_df['Improvement'] = results_df['Fine-Tuned (LoRA)'] - results_df['Zero-Shot (Base)']

# Formatting for display
print(results_df.round(4).to_markdown(index=False))

print("\nDetailed Confusion Matrix Comparison:")
print("\n--- Zero-Shot Confusion Matrix ---")
print(confusion_matrix(y_true, y_pred_zero, labels=VALID_LABELS))
print("\n--- Fine-Tuned Confusion Matrix ---")
print(confusion_matrix(y_true, y_pred, labels=VALID_LABELS))